# Who Actually Buys Your Crops? Town Demand as the Hidden Half of the MarketMost Kaggriculture strategy talk is about the **supply** side: what to plant, how manyanimals, how fast to expand. This notebook is about the **demand** side, which turns outto decide more of the game than the crop table does.The market price of every product is a function of one number — the market's inventory ofit. Two forces move that number:1. **Players selling** pushes inventory up, price down.2. **The town consuming** pulls inventory down, price up — *permanently*.The second force is the one that is easy to miss, because it never appears in yourobservation as an event. It just quietly happens every few turns. And it is notdistributed evenly: some products have a town eating them faster than two farms canpossibly supply, and others have **no town demand whatsoever**.Everything below is computed from the shipped engine (`kaggle_environments.envs.kaggriculture`),not transcribed from the README, so you can re-run it and check me.*I am a high-school student doing this competition as an economics project. If I have gotsomething wrong, please say so in the comments — I would rather be corrected than upvoted.*

In [ ]:
import math
from kaggle_environments.envs.kaggriculture.kaggriculture import (
    SHOPS, PRODUCTS, TOWN_CENTER_PRODUCTS, TOWN_CENTER_DEMAND_SCHEDULE,
    MARKET_PARAMS, MARKET_I0, market_price, CROPS, ANIMALS,
)

TURNS_PER_DAY, DAYS = 24, 30
SHOP_SELL_INTERVAL   = 4    # every 4 turns  -> 6 ticks/day
CENTER_SELL_INTERVAL = 12   # every 12 turns -> 2 ticks/day

print("Shops in the game:")
for name, prods in SHOPS.items():
    tag = "  (single-product -> consumes 2x)" if len(prods) == 1 else ""
    print(f"  {name:<16} {', '.join(prods)}{tag}")

## 1. How much does the town eat per day?Each unlocked shop consumes one of every product it demands every `townShopSellInterval`(4) turns — so **6 units per day per product**, doubled for single-product shops. The towncentre separately consumes one of *every* product (except fertilizer) every`townCenterSellInterval` (12) turns, and that rate scales: **x1 before day 10, x2 from day10, x4 from day 20**.

In [ ]:
shop_per_day = {p: 0 for p in PRODUCTS}
for shop, prods in SHOPS.items():
    mult = 2 if len(prods) == 1 else 1
    for p in prods:
        shop_per_day[p] += (TURNS_PER_DAY // SHOP_SELL_INTERVAL) * mult

n_shops_wanting = {p: sum(1 for s in SHOPS.values() if p in s) for p in PRODUCTS}

print(f"{'product':<12} {'shops':>6} {'shop units/day':>15} {'town-centre/day (d0/d10/d20)':>30}")
print("-" * 68)
for p in PRODUCTS:
    if p in TOWN_CENTER_PRODUCTS:
        base = TURNS_PER_DAY // CENTER_SELL_INTERVAL      # 2 ticks/day
        centre = f"{base} / {base*2} / {base*4}"
    else:
        centre = "none - excluded"
    print(f"{p:<12} {n_shops_wanting[p]:>6} {shop_per_day[p]:>15} {centre:>30}")

## 2. The two products nobody is buyingLook at the bottom of that table.- **MELON has zero shop demand.** No shop in the game lists it. Its only buyer is the town  centre's 2/4/8 per day — 140 units across the whole season.- **FERTILIZER has zero demand, full stop.** No shop wants it, *and* it is explicitly  excluded from the town centre's list (`TOWN_CENTER_PRODUCTS = [p for p in PRODUCTS if p != "FERTILIZER"]`).That second one has a consequence I have not seen discussed much. Every other product'smarket inventory goes both up (you sell) and down (the town eats). Fertilizer's inventory**only ever goes up**. There is no mechanism in the game that removes a fertilizer from themarket. So once fertilizer is glutted, it is glutted *for the rest of the season* — theprice cannot recover, for either player.

In [ ]:
season_town_demand = {}
for p in PRODUCTS:
    total = 0
    # shops unlock every 3 days; assume the i-th unlock is active from that day onward
    unlock_days = [d for d in range(1, DAYS + 1) if d % 3 == 0]
    shops_sorted = list(SHOPS.items())
    for i, (shop, prods) in enumerate(shops_sorted):
        if i >= len(unlock_days) or p not in prods:
            continue
        mult = 2 if len(prods) == 1 else 1
        active_days = DAYS - unlock_days[i]
        total += active_days * (TURNS_PER_DAY // SHOP_SELL_INTERVAL) * mult
    if p in TOWN_CENTER_PRODUCTS:
        for d in range(DAYS):
            m = next(mm for thr, mm in TOWN_CENTER_DEMAND_SCHEDULE if d >= thr)
            total += (TURNS_PER_DAY // CENTER_SELL_INTERVAL) * m
    season_town_demand[p] = total

print(f"{'product':<12} {'town eats/season':>18} {'price if you sell NOTHING':>27}")
print("-" * 60)
for p in sorted(PRODUCTS, key=lambda x: -season_town_demand[x]):
    px = market_price(p, MARKET_I0 - season_town_demand[p])
    base = MARKET_PARAMS[p]["base"]
    print(f"{p:<12} {season_town_demand[p]:>18} {px:>18} ({px/base:>4.2f}x base)")

## 3. Why this changes what you plantA product's price is not a fixed property of the crop. It is the result of a race betweenyour selling and the town's eating. That gives three regimes:| Regime | Condition | What to do ||---|---|---|| **Demand-rich** | town eats faster than both farms can supply | scale it — the price *rises* all season || **Balanced** | comparable rates | drip-sell at roughly the town's rate || **Demand-dead** | little or no town demand | take a fixed budget of units, then stop |Melon sits squarely in demand-dead, and it also has the harshest glut curve in the game(`above_func = "sq"`, `above_target = 3.60`). That combination is why "just plant allmelons" reliably disappoints: the first tranche is the best-paying thing on the board, andthe tranche after that is nearly worthless.

In [ ]:
def revenue_curve(item, qty):
    """Cumulative revenue from selling `qty` units into your own price impact."""
    inv, total, out = MARKET_I0, 0, []
    for i in range(qty):
        px = market_price(item, inv)
        total += px
        if px > 1:          # sales at the $1 floor do not add to market inventory
            inv += 1
        out.append(total)
    return out

print(f"{'product':<12} " + " ".join(f"{f'Q={q}':>9}" for q in (25, 50, 100, 200, 400)))
print("-" * 62)
for p in ["MELON", "WOOL", "MILK", "STRAWBERRY", "TOMATO", "CARROT", "FERTILIZER", "WHEAT", "EGG"]:
    c = revenue_curve(p, 400)
    print(f"{p:<12} " + " ".join(f"{c[q-1]:>9,}" for q in (25, 50, 100, 200, 400)))
print()
print("Read the rows across: where the number stops growing is where that product dies.")
print("MELON earns ~21.7k on its first 100 units and only ~5k more on the next 100.")
print("EGG is the only product still paying roughly full price at Q=400.")

## 4. A selling rule that follows from thisIf prices recover between sales (because the town keeps eating), then dumping is strictlyworse than dripping — you are handing yourself the bottom of your own price curve.The rule I use is a **reserve price**: sell a unit only while the *marginal* price of thatunit stays above a floor. It is a two-line binary search because `market_price` is monotonein inventory, and it automatically adapts to what the opponent has already dumped — if theyhave crashed wool, my agent simply stops selling wool and waits for the town to drain itback.

In [ ]:
def sellable_qty(item, market_inventory, have, reserve):
    """Largest n <= have such that all n units still price at >= reserve."""
    lo, hi = 0, have
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if market_price(item, market_inventory + mid - 1) >= reserve:
            lo = mid
        else:
            hi = mid - 1
    return lo

# You are holding 60 wool. How many should you sell right now?
for opp_dumped in (0, 40, 100):
    inv = MARKET_I0 + opp_dumped
    n = sellable_qty("WOOL", inv, 60, reserve=95)
    print(f"opponent already sold {opp_dumped:>3} wool -> price ${market_price('WOOL', inv):>3}"
          f" -> sell {n:>2} of my 60 now, hold the rest")

## 5. Caveats, and what I am not claiming- The season-total demand numbers assume all eight shops unlock, and shop unlock order is  **random per episode**, so per-product totals vary game to game. The *direction* is  stable; the exact figure is not.- I have deliberately not published my agent's parameters. The analysis above is the part  that is useful to everyone; the tuning is the part I am still competing with. I would  rather be honest about that than pretend this is the whole picture.- Everything here is arithmetic on the engine's constants. It says what the market *will*  pay. It says nothing about whether your farm can actually produce it — action economy is  a separate and probably larger problem, and it is where I have lost the most.### Credit where it is dueThe observation that melon lacks shop demand is not original to me — it appears in publicwork by **Rayk Kretzschmar** (*Findings from Zero to Top Meta*), **Roman Rozen**, and**Timass** (*The Kaggriculture Market, End to End*), among others. What I have tried to addis the computation from the engine's own tables, the season-long demand totals, and thefertilizer point, which I have not seen stated elsewhere. Please go upvote their notebooks.If you spot an error, tell me — I will fix it and credit you.